# Embeddings & Encoding Basics

**Purpose:** Encoding is the heart. Students should deeply understand:

> **Meaning comes from the encoder.**

In this notebook, we will:
1. Define what an embedding is (intuition).
2. Load a tiny corpus (~20 sentences).
3. Encode the corpus with a fast sentence-transformer model.
4. Inspect vectors (shape, a few values).
5. Compute pairwise similarity (similarity matrix).
6. Find top similar pairs.
7. Reflect: *Which pairs surprised you? Why?*


## 1) What is an embedding?

An **embedding** is a numeric vector representation of something (text, image, audio, etc.) such that:

- Similar meanings → **vectors close together**
- Different meanings → **vectors far apart**

**Intuition:** an embedding is like a coordinate in *meaning-space*:
- Each sentence becomes a point in a high-dimensional space.
- You can't interpret a single number directly, but you *can* compare vectors.

We'll use **cosine similarity** to measure "closeness" in this meaning-space.


## 2) Setup: install & import

We'll use a small, fast sentence-transformer model:
- `all-MiniLM-L6-v2` (commonly used for demos and learning)

If `sentence-transformers` isn't installed, the cell will install it.


In [17]:
# If needed, install sentence-transformers (safe to re-run)
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    !uv add sentence-transformers

In [10]:
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)

## 3) Load a tiny corpus (~20 sentences)

We want a **varied** set so we can see interesting similarities and surprises.


In [11]:
sentences = [
    "I love eating mangoes during summer.",
    "The stock market fell sharply after the earnings report.",
    "Machine learning models can recognize patterns in data.",
    "My dog chased a squirrel across the park.",
    "We should refactor the code to reduce technical debt.",
    "The chef prepared a spicy bowl of ramen.",
    "Artificial intelligence is transforming many industries.",
    "A typhoon is expected to make landfall tomorrow evening.",
    "She bought gasoline before driving to the province.",
    "The athlete trained daily to improve endurance.",
    "Banks assess credit risk before approving loans.",
    "The new smartphone has an impressive battery life.",
    "He studied calculus to understand optimization.",
    "The teacher explained photosynthesis to the class.",
    "We ran SQL queries to validate the dataset.",
    "The concert was loud, energetic, and unforgettable.",
    "I enjoy reading novels on rainy afternoons.",
    "The company launched a loyalty program to retain customers.",
    "Coffee helps me stay focused during late-night work.",
    "Traffic on the expressway was heavy due to an accident.",
]

len(sentences)

20

## 4) Encode with a sentence transformer

Remember the key idea:

> **Meaning comes from the encoder.**

Different encoders (models) create different "meaning-spaces".


In [12]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

embeddings = model.encode(sentences, normalize_embeddings=True)  # normalize helps cosine similarity
embeddings.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(20, 384)

### Inspect a vector

Vectors are high-dimensional, and single values are **not directly interpretable**.
What matters is how vectors compare to each other.

We'll print:
- the embedding shape
- the first 5 values of the first sentence embedding


In [13]:
print("Embedding matrix shape:", embeddings.shape)
print("\nFirst sentence:")
print(" ", sentences[0])
print("\nFirst embedding (first 5 values):")
print(embeddings[0][:5])

Embedding matrix shape: (20, 384)

First sentence:
  I love eating mangoes during summer.

First embedding (first 5 values):
[ 0.00060364 -0.01296409  0.05955993  0.08697072  0.02022286]


## 5) Pairwise similarity

We'll compute a **similarity matrix** where entry *(i, j)* is cosine similarity between sentence *i* and sentence *j*.

- 1.0 → identical direction (very similar)
- 0.0 → orthogonal (unrelated)
- -1.0 → opposite direction (rare in sentence embeddings, but possible)


In [14]:
sim = cosine_similarity(embeddings, embeddings)

sim_df = pd.DataFrame(sim, index=[f"S{i:02d}" for i in range(len(sentences))],
                      columns=[f"S{i:02d}" for i in range(len(sentences))])

sim_df.head()

,S00,S01,S02,S03,S04,S05,S06,S07,S08,S09,S10,S11,S12,S13,S14,S15,S16,S17,S18,S19
S00,1.000000,-0.044335,0.144239,0.069895,-0.043211,0.160249,-0.013761,0.073763,0.037717,0.101745,-0.031711,0.072471,-0.012373,0.112608,0.061795,0.061463,0.293309,0.057497,0.199893,-0.086751
S01,-0.044335,1.000000,-0.014369,0.137937,0.093846,-0.081971,0.107865,-0.028823,-0.041234,-0.038832,0.010775,-0.020070,0.035236,0.033032,0.064059,0.075986,0.078157,0.029449,0.007155,0.111255
S02,0.144239,-0.014369,1.000000,-0.032503,0.093475,-0.000107,0.349384,-0.004677,-0.014776,0.101401,0.100215,0.042588,0.159358,0.112091,0.365353,-0.052309,0.048620,0.121060,0.101656,0.001791
S03,0.069895,0.137937,-0.032503,1.000000,-0.058561,-0.021763,-0.051744,-0.010634,0.062767,0.067037,-0.043509,0.044317,0.065419,-0.016302,-0.042036,0.144520,0.078146,0.035015,0.007294,0.191811
S04,-0.043211,0.093846,0.093475,-0.058561,1.000000,-0.032331,0.296142,0.047545,0.026167,0.011668,0.224476,0.139520,0.094468,0.001555,0.114448,-0.101779,-0.009270,0.123866,0.031852,0.000268


### Pretty view: sentence lookup table

This helps interpret the similarity matrix.


In [15]:
lookup = pd.DataFrame({
    "id": [f"S{i:02d}" for i in range(len(sentences))],
    "sentence": sentences
})
lookup

,id,sentence
0,S00,I love eating mangoes during summer.
1,S01,The stock market fell sharply after the earnings report.
2,S02,Machine learning models can recognize patterns in data.
3,S03,My dog chased a squirrel across the park.
4,S04,We should refactor the code to reduce technical debt.
5,S05,The chef prepared a spicy bowl of ramen.
6,S06,Artificial intelligence is transforming many industries.
7,S07,A typhoon is expected to make landfall tomorrow evening.
8,S08,She bought gasoline before driving to the province.
9,S09,The athlete trained daily to improve endurance.


## 6) Top similar pairs (excluding self-matches)

We'll find the most similar sentence pairs.


In [16]:
# Build all (i, j) pairs where i < j
pairs = []
n = len(sentences)
for i in range(n):
    for j in range(i + 1, n):
        pairs.append((i, j, sim[i, j]))

pairs_sorted = sorted(pairs, key=lambda x: x[2], reverse=True)

top_k = 10
top_pairs = []
for i, j, score in pairs_sorted[:top_k]:
    top_pairs.append({
        "pair": f"S{i:02d}–S{j:02d}",
        "similarity": float(score),
        "sentence_a": sentences[i],
        "sentence_b": sentences[j],
    })

top_pairs_df = pd.DataFrame(top_pairs)
top_pairs_df

,pair,similarity,sentence_a,sentence_b
0,S02–S14,0.365353,Machine learning models can recognize patterns in data.,We ran SQL queries to validate the dataset.
1,S02–S06,0.349384,Machine learning models can recognize patterns in data.,Artificial intelligence is transforming many industries.
2,S04–S06,0.296142,We should refactor the code to reduce technical debt.,Artificial intelligence is transforming many industries.
3,S00–S16,0.293309,I love eating mangoes during summer.,I enjoy reading novels on rainy afternoons.
4,S16–S18,0.287522,I enjoy reading novels on rainy afternoons.,Coffee helps me stay focused during late-night work.
5,S15–S19,0.246252,"The concert was loud, energetic, and unforgettable.",Traffic on the expressway was heavy due to an accident.
6,S12–S13,0.227367,He studied calculus to understand optimization.,The teacher explained photosynthesis to the class.
7,S04–S10,0.224476,We should refactor the code to reduce technical debt.,Banks assess credit risk before approving loans.
8,S08–S19,0.214443,She bought gasoline before driving to the province.,Traffic on the expressway was heavy due to an accident.
9,S09–S12,0.206450,The athlete trained daily to improve endurance.,He studied calculus to understand optimization.


## 7) Mini reflection

1. Which top similar pair surprised you the most?
2. Why do you think the encoder grouped them together?
3. Pick two sentences that *you think* should be similar but are not near the top—what might explain that?
4. If we switched to a different encoder model, which pairs might change?

> **Key takeaway:** The embedding space is defined by the encoder (the model).


## Optional extension: Try a different encoder

Change `model_name` to another sentence-transformer and re-run:
- Compare top similar pairs
- Observe which groupings change

Example alternatives:
- `sentence-transformers/paraphrase-MiniLM-L3-v2` (even smaller)
- `sentence-transformers/all-mpnet-base-v2` (stronger but slower)
